In [10]:
import json
from datasets import load_dataset
from transformers import pipeline

# Load emotion detection model (Emotion classifier)
emotion_classifier = pipeline("text-classification", model="bhadresh-savani/bert-base-uncased-emotion", tokenizer="bhadresh-savani/bert-base-uncased-emotion")

# Updated disorder classification for the six specific types with expanded keywords
def classify_disorder(text):
    """
    Classifies the input text into one of the six disorder types:
    Bipolar Disorder, BPD, Anxiety, Depression, Schizophrenia, General Mental Illness.
    """
    disorder_keywords = {
        "Bipolar Disorder": [
            "bipolar", "manic", "mania", "mood swings", "high energy", "euphoric",
            "hyperactive", "reckless behavior", "grandiosity", "impulsive", "risky behavior",
            "lack of sleep", "irritable mood", "depressed mood"
        ],
        "BPD": [
            "borderline", "BPD", "emotionally unstable", "self-harm", "impulsive", "fear of abandonment",
            "black and white thinking", "rapid mood swings", "unstable relationships", "emotional dysregulation",
            "feelings of emptiness", "identity disturbance", "suicidal ideation", "reckless behavior"
        ],
        "Anxiety": [
            "anxiety", "panic", "nervous", "worry", "overthinking", "fear", "constant worrying",
            "racing thoughts", "tension", "nervousness", "avoidance", "social anxiety",
            "heart palpitations", "sweating", "shortness of breath", "restlessness"
        ],
        "Depression": [
            "depressed", "sad", "hopeless", "down", "low mood", "crying", "fatigue", "lack of energy",
            "loss of interest", "feelings of worthlessness", "feeling numb", "guilt", "negative thoughts",
            "insomnia", "sleeping too much", "appetite changes", "suicidal ideation", "helplessness"
        ],
        "Schizophrenia": [
            "schizophrenia", "hallucinations", "delusions", "psychosis", "paranoia", "delusional thoughts",
            "voices in my head", "visual hallucinations", "auditory hallucinations", "disorganized speech",
            "catatonia", "disconnected thoughts", "detachment from reality", "disorganized behavior"
        ],
        "General Mental Illness": [
            "mental illness", "struggling", "not feeling right", "overwhelmed", "numb", "irritable",
            "out of control", "distressed", "not okay", "unmotivated", "exhausted", "burnout", "emotional distress",
            "personal issues", "feeling stuck", "feeling lost", "confused", "unable to cope", "feeling empty"
        ]
    }

    # Convert text to lower case for matching
    text_lower = text.lower()
    
    # Check each disorder category
    for disorder, keywords in disorder_keywords.items():
        if any(keyword in text_lower for keyword in keywords):
            return disorder
    
    # Default to General Mental Illness if no specific match
    return "General Mental Illness"

# Function to automatically classify emotion and disorder type for each entry in the dataset
def process_and_classify_entry(user_input, existing_disorder=None):
    """
    Classify emotion and disorder type based on user input.
    
    Args:
    - user_input (str): The user input to classify.
    - existing_disorder (str): The existing disorder label, if available (optional).
    
    Returns:
    - emotion (str): The detected emotion from the user input.
    - disorder_type (str): The detected disorder type based on keywords or existing label.
    """
    # Truncate input to fit within BERT's max length (512 tokens)
    max_length = 512
    tokenizer = emotion_classifier.tokenizer
    tokens = tokenizer.encode(user_input, truncation=True, max_length=max_length, return_tensors="pt")
    truncated_input = tokenizer.decode(tokens[0], skip_special_tokens=True)
    
    # Emotion detection (e.g., sadness, joy, fear)
    emotions = emotion_classifier(truncated_input)
    detected_emotion = emotions[0]['label']
    
    # Use existing disorder if available, otherwise classify it
    detected_disorder = existing_disorder if existing_disorder else classify_disorder(user_input)
    
    return detected_emotion, detected_disorder

# Load and process datasets
def process_multiple_datasets(dataset_names, system_prompt, output_file):
    """
    Automates the conversion of multiple datasets into OpenAI fine-tuning JSONL format with added behavioral expectations.
    
    Args:
    - dataset_names (list): List of dataset names or file paths to custom datasets.
    - system_prompt (str): The updated system prompt with behavior expectations.
    - output_file (str): File path to save the final JSONL fine-tuning dataset.
    """
    # Final dataset collection
    final_dataset = []
    
    for dataset_name in dataset_names:
        # Load the dataset
        dataset = load_dataset(dataset_name)

        # Check if the dataset contains a 'train' split and extract data
        if 'train' in dataset:
            dataset = dataset['train']
        elif 'test' in dataset:
            dataset = dataset['test']
        else:
            print(f"⚠️ Dataset {dataset_name} does not contain 'train' or 'test' split. Skipping.")
            continue
        
        # Process each entry in the dataset
        for item in dataset:
            # Check if the dataset is a dictionary, otherwise continue to next item
            if isinstance(item, dict):
                user_input = item.get("user", item.get("Context", "").strip())
                assistant_response = item.get("assistant", item.get("Response", "").strip())
                existing_disorder = item.get("disorder_type", None)  # Use the existing disorder if available
                
                if user_input and assistant_response:
                    # Classify emotion and disorder type automatically, or use existing disorder
                    detected_emotion, detected_disorder = process_and_classify_entry(user_input, existing_disorder)
                    
                    # Build the training entry with the system prompt, user input, and assistant response
                    final_dataset.append({
                        "messages": [
                            {"role": "system", "content": system_prompt},
                            {"role": "user", "content": user_input},
                            {"role": "assistant", "content": assistant_response},
                            {"role": "system", "content": f"Detected Emotion: {detected_emotion}, Detected Disorder Type: {detected_disorder}"}
                        ]
                    })
            else:
                print(f"Skipping invalid entry: {item}")
    
    # Save the dataset to a JSONL file
    with open(output_file, "w") as f:
        for entry in final_dataset:
            f.write(json.dumps(entry) + "\n")
    
    print(f"✅ Dataset saved as {output_file} with {len(final_dataset)} entries.")

# Define the system prompt with emotion detection, disorder type, differential diagnosis, and CBT
system_prompt = """
You are ChillBuddy — a Gen Z-friendly, emotionally intelligent AI mental health counselor trained in CBT using cognitive reframing.

Your responsibilities include:
1. Detecting and labeling the user's **emotion** (e.g., sadness, anxiety, anger, numbness).
2. Identifying potential **primary mental health conditions** (e.g., Bipolar Disorder, BPD, Anxiety, Depression, Schizophrenia, General Mental Illness).
3. Suggesting a possible **differential diagnosis** if symptoms could overlap or be unclear.
4. Responding with **CBT-based persuasive counseling**, especially using cognitive reframing.
5. Communicating in a **casual, relatable Gen Z tone** (e.g., "fr", "lowkey", emojis).
6. Engaging in **casual/general conversation** when users send greetings or friendly banter.
7. Generating 5 internal responses to every message, scoring them (on empathy, helpfulness, CBT accuracy, Gen Z tone), and replying with the highest-scoring one.

Be kind, validating, and help the user reframe their thoughts and feelings. Guide them with empathy, not judgment.
"""

# Example of using the function with Hugging Face datasets
output_file = "gpt4o_mini_finetune_mental_health_with_disorders.jsonl"
dataset_names = ["nbertagnolli/counsel-chat", "Amod/mental_health_counseling_conversations"]
process_multiple_datasets(dataset_names, system_prompt, output_file)

Device set to use mps:0
Repo card metadata block was not found. Setting CardData to empty.


✅ Dataset saved as gpt4o_mini_finetune_mental_health_with_disorders.jsonl with 3508 entries.
